# ZD3D — Eikonal Weight Ablation

Sweeps the eikonal loss weight over six values and three random seeds
(18 training runs) to quantify sensitivity to this hyperparameter and select
the value used in `02_train_and_evaluate.ipynb`.

**This benchmark is different from the plain-sphere ones.** The Zalesak initial
field is the CSG difference `max(phi_sphere, -phi_slot)`, which is *not* an
exact signed-distance function near the slot's reentrant edges: it returns the
distance to the nearer face rather than to the edge, so `|grad phi| != 1` there
**in the exact solution itself**. The eikonal term therefore competes with the
initial-condition term near the slot, rather than reinforcing it as it does for
a smooth sphere. Expect the selected weight to be much smaller here, and expect
`w_eik = 1.0` to be actively harmful.

Note that Zalesak is solid-body rotation — a rigid motion — not a deformation
test. It probes preservation of corners and thin features under rotation.

**Runtime:** longer per run than the sphere benchmarks (Fourier features,
RAD/RAR resampling, causal weighting). Set `EST_MIN_PER_RUN` from your first
measured run. Results are appended to CSV after every run and completed runs
are skipped on restart, so a disconnect costs at most one run.

**Outputs:** `ZD3D_eikonal_ablation.csv` with per-snapshot and averaged metrics
for every run, plus a summary table with seed spread.

In [1]:
# --- setup -------------------------------------------------------------
import sys, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/ZD3D'
except ImportError:
    PROJECT_DIR = os.getcwd()

sys.path.insert(0, PROJECT_DIR)
os.makedirs(PROJECT_DIR, exist_ok=True)

import csv, itertools, time
import numpy as np
import torch
import zd3d_core as core

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
device: cuda


In [2]:
# --- configuration -----------------------------------------------------
W_EIK_GRID   = [0.0, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]
SEEDS        = [42, 43, 44]

W_PDE, W_IC  = 1.0, 10.0
N_F, N_I     = 20000, 10000
LAYERS, NEURONS = 8, 256
N_FOURIER, B_SCALE = 128, 5.0
T_FINAL      = 2 * np.pi
ADAM_EPOCHS  = 20000        # same budget as the reported run, so the ranking
LBFGS_EPOCHS = 500          # transfers without extrapolation
GRID_N       = 201
N_SNAPSHOTS  = 6

M_CHUNKS, EPS_CAUSAL = 32, 1.0
RAD_FREQ, RAD_CANDIDATES, RAD_K, RAD_C = 1000, 20000, 1.0, 1.0
RAR_FREQ, RAR_CANDIDATES, RAR_ADD      = 1800, 50000, 500

EST_MIN_PER_RUN = 15        # update from your first measured run
CSV_PATH = f'{PROJECT_DIR}/ZD3D_eikonal_ablation_20K.csv'

times, tlabels = core.snapshot_times(T_FINAL, N_SNAPSHOTS)
grid = core.Grid(n=GRID_N)
METRICS = ['rel_L2', 'vol_MAPE', 'eik_dev', 'l2_abs', 'vol_abs', 'vol_floor']

FIELDS = ['w_eik', 'seed']
for m in METRICS:
    FIELDS += [f'{m}_t{i}' for i in range(N_SNAPSHOTS)] + [f'avg_{m}']
FIELDS += ['loss_adam', 'loss_final', 'lbfgs_gain', 'n_points', 'minutes']

print(f'{len(W_EIK_GRID) * len(SEEDS)} runs, {len(FIELDS)} columns per row')

18 runs, 49 columns per row


In [3]:
# --- one run -----------------------------------------------------------
def run_one(w_eik, seed):
    net = core.build_model(device, seed=seed, layers=LAYERS, neurons=NEURONS,
                           n_fourier=N_FOURIER, b_scale=B_SCALE)
    X_f, X_i = core.sample_points(N_F, N_I, device, seed=seed)
    sampler = core.AdaptiveSampler(net, X_f, T_FINAL, device,
                                   rad_freq=RAD_FREQ, rad_candidates=RAD_CANDIDATES,
                                   rad_k=RAD_K, rad_c=RAD_C,
                                   rar_freq=RAR_FREQ, rar_candidates=RAR_CANDIDATES,
                                   rar_add=RAR_ADD)
    loss_fn = core.make_loss(net, sampler, X_i, T_FINAL, W_PDE, W_IC, w_eik,
                             m_chunks=M_CHUNKS, eps_causal=EPS_CAUSAL)

    tr = core.train(net, loss_fn, sampler=sampler, adam_epochs=ADAM_EPOCHS,
                    lbfgs_epochs=LBFGS_EPOCHS, verbose=False)
    res = core.evaluate(net, grid, times, T_FINAL, device, cache=False)

    row = {'w_eik': w_eik, 'seed': seed}
    for m in METRICS:
        for i in range(N_SNAPSHOTS):
            row[f'{m}_t{i}'] = f'{res[m][i]:.6e}'
        row[f'avg_{m}'] = f"{res['avg_' + m]:.6e}"
    row.update({'loss_adam': f"{tr['loss_adam']:.6e}",
                'loss_final': f"{tr['loss_final']:.6e}",
                'lbfgs_gain': f"{tr['lbfgs_gain']:.6e}",
                'n_points': tr['n_points'],
                'minutes': f"{tr['minutes']:.2f}"})

    del net, X_f, X_i, sampler
    if device == 'cuda':
        torch.cuda.empty_cache()
    return row

In [4]:
# --- sweep (resumable) -------------------------------------------------
done = set()
if os.path.exists(CSV_PATH):
    with open(CSV_PATH) as f:
        for r in csv.DictReader(f):
            done.add((float(r['w_eik']), int(r['seed'])))
    print(f'{len(done)} run(s) already complete — skipping those')

todo = [p for p in itertools.product(W_EIK_GRID, SEEDS) if p not in done]
print(f'{len(todo)} run(s) to go, ~{len(todo) * EST_MIN_PER_RUN / 60:.1f} h\n', flush=True)

for k, (wv, sd) in enumerate(todo, 1):
    print(f'[{k}/{len(todo)}] w_eik={wv:g} seed={sd} ...', flush=True)
    row = run_one(wv, sd)
    new = not os.path.exists(CSV_PATH) or os.path.getsize(CSV_PATH) == 0
    with open(CSV_PATH, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if new:
            w.writeheader()
        w.writerow(row)
    print(f"    relL2={float(row['avg_rel_L2']):.4f}%  "
          f"MAPE={float(row['avg_vol_MAPE']):.4f}%  "
          f"eik={float(row['avg_eik_dev']):.3e}  "
          f"lbfgs_gain={float(row['lbfgs_gain']):.2e}  "
          f"({float(row['minutes']):.1f} min)", flush=True)

18 run(s) already complete — skipping those
0 run(s) to go, ~0.0 h



## Summary and selection

Decision rule, fixed before the sweep:

1. **Primary** — lowest average relative L₂
2. **Tiebreak 1** — lowest average volume MAPE
3. **Tiebreak 2** — lowest eikonal deviation
4. **Tiebreak 3** — the smaller weight

Two weights count as *tied* when the gap between their means is smaller than
the larger of their two seed standard deviations. Eikonal deviation is only a
second-order tiebreak on purpose: it is essentially the quantity the swept term
minimises, so ranking by it would recover the sweep by construction.

In [5]:
# --- summary -----------------------------------------------------------
rows = list(csv.DictReader(open(CSV_PATH)))
agg = {}
for r in rows:
    agg.setdefault(float(r['w_eik']), []).append(r)

stats = {}
print(f"{'w_eik':>8} | {'rel L2 (%)':>19} | {'vol MAPE (%)':>19} | "
      f"{'eik dev':>11} | {'n':>2}")
print('-' * 74)
for wv in sorted(agg):
    g = agg[wv]
    m = {k: np.mean([float(r['avg_' + k]) for r in g]) for k in
         ('rel_L2', 'vol_MAPE', 'eik_dev')}
    s = {k: np.std([float(r['avg_' + k]) for r in g]) for k in
         ('rel_L2', 'vol_MAPE', 'eik_dev')}
    stats[wv] = (m, s, len(g))
    print(f"{wv:>8g} | {m['rel_L2']:>9.4f} ± {s['rel_L2']:<7.4f} | "
          f"{m['vol_MAPE']:>9.4f} ± {s['vol_MAPE']:<7.4f} | "
          f"{m['eik_dev']:>11.3e} | {len(g):>2}")

def tied(a, b, key):
    (ma, sa, _), (mb, sb, _) = stats[a], stats[b]
    return abs(ma[key] - mb[key]) < max(sa[key], sb[key])

best = min(stats, key=lambda w: stats[w][0]['rel_L2'])
for cand in sorted(stats, key=lambda w: stats[w][0]['rel_L2'])[1:]:
    if tied(best, cand, 'rel_L2'):
        if not tied(best, cand, 'vol_MAPE'):
            best = min(best, cand, key=lambda w: stats[w][0]['vol_MAPE'])
        elif not tied(best, cand, 'eik_dev'):
            best = min(best, cand, key=lambda w: stats[w][0]['eik_dev'])
        else:
            best = min(best, cand)

ties = sorted(w for w in stats if w != best and tied(best, w, 'rel_L2'))
print('-' * 74)
print(f'selected w_eik = {best:g}')
if ties:
    print('statistically tied on rel L2 with: ' + ', '.join(f'{w:g}' for w in ties))
    print('-> report insensitivity across this range rather than a single value')
print(f"mean L-BFGS gain over all runs: "
      f"{np.mean([float(r['lbfgs_gain']) for r in rows]):.3e}")

   w_eik |          rel L2 (%) |        vol MAPE (%) |     eik dev |  n
--------------------------------------------------------------------------
       0 |    2.4125 ± 0.1099  |    3.1026 ± 1.8654  |   1.742e-01 |  3
   1e-05 |    2.3207 ± 0.1362  |    2.7265 ± 0.9485  |   1.915e-01 |  3
  0.0001 |    1.8767 ± 0.1951  |    1.8584 ± 0.9304  |   1.621e-01 |  3
   0.001 |    0.5163 ± 0.1051  |    1.3371 ± 0.8625  |   1.238e-01 |  3
    0.01 |    1.0314 ± 0.0586  |   16.1529 ± 1.2415  |   6.409e-02 |  3
     0.1 |    1.0631 ± 0.0229  |   17.9838 ± 0.3562  |   5.190e-02 |  3
--------------------------------------------------------------------------
selected w_eik = 0.001
mean L-BFGS gain over all runs: 2.211e-05


## Error growth per snapshot

Per-snapshot averages across seeds, for the time-evolution table/figure.

In [6]:
# --- per-snapshot table ------------------------------------------------
METRIC = 'rel_L2'      # 'rel_L2' | 'vol_MAPE' | 'eik_dev' | 'l2_abs'

hdr = ' | '.join(f'{lab:>9}' for lab in tlabels)
print(f'{METRIC} by snapshot (mean over seeds)\n')
print(f"{'w_eik':>8} | {hdr}")
print('-' * (11 + 12 * len(tlabels)))
for wv in sorted(agg):
    vals = [np.mean([float(r[f'{METRIC}_t{i}']) for r in agg[wv]])
            for i in range(N_SNAPSHOTS)]
    print(f'{wv:>8g} | ' + ' | '.join(f'{v:>9.4f}' for v in vals))

rel_L2 by snapshot (mean over seeds)

   w_eik |         0 |       T/5 |      2T/5 |      3T/5 |      4T/5 |         T
-----------------------------------------------------------------------------------
       0 |    0.1269 |    1.9162 |    2.8546 |    2.9901 |    3.1643 |    3.4228
   1e-05 |    0.1321 |    1.8406 |    2.7740 |    2.8669 |    2.9651 |    3.3453
  0.0001 |    0.1203 |    1.4065 |    2.2123 |    2.1751 |    2.4214 |    2.9245
   0.001 |    0.1350 |    0.3873 |    0.4185 |    0.5588 |    0.6027 |    0.9952
    0.01 |    0.5489 |    1.0829 |    1.1070 |    1.1218 |    1.1265 |    1.2010
     0.1 |    0.5163 |    1.1484 |    1.1643 |    1.1797 |    1.1561 |    1.2136


In [7]:
# --- per-snapshot table ------------------------------------------------
METRIC = 'vol_MAPE'      # 'rel_L2' | 'vol_MAPE' | 'eik_dev' | 'l2_abs'

hdr = ' | '.join(f'{lab:>9}' for lab in tlabels)
print(f'{METRIC} by snapshot (mean over seeds)\n')
print(f"{'w_eik':>8} | {hdr}")
print('-' * (11 + 12 * len(tlabels)))
for wv in sorted(agg):
    vals = [np.mean([float(r[f'{METRIC}_t{i}']) for r in agg[wv]])
            for i in range(N_SNAPSHOTS)]
    print(f'{wv:>8g} | ' + ' | '.join(f'{v:>9.4f}' for v in vals))

vol_MAPE by snapshot (mean over seeds)

   w_eik |         0 |       T/5 |      2T/5 |      3T/5 |      4T/5 |         T
-----------------------------------------------------------------------------------
       0 |    0.7077 |    1.5679 |    2.8617 |    3.9270 |    4.5831 |    4.9683
   1e-05 |    1.0968 |    1.4455 |    2.5498 |    3.3990 |    3.8245 |    4.0434
  0.0001 |    0.4517 |    0.3519 |    1.2761 |    2.4454 |    3.1387 |    3.4863
   0.001 |    1.1767 |    0.9978 |    1.2935 |    1.3607 |    1.3921 |    1.8021
    0.01 |    8.7588 |   17.3726 |   17.3284 |   17.5176 |   17.9007 |   18.0390
     0.1 |   10.3497 |   19.4265 |   19.5153 |   19.6066 |   19.6090 |   19.3958


In [8]:
# --- per-snapshot table ------------------------------------------------
METRIC = 'eik_dev'      # 'rel_L2' | 'vol_MAPE' | 'eik_dev' | 'l2_abs'

hdr = ' | '.join(f'{lab:>9}' for lab in tlabels)
print(f'{METRIC} by snapshot (mean over seeds)\n')
print(f"{'w_eik':>8} | {hdr}")
print('-' * (11 + 12 * len(tlabels)))
for wv in sorted(agg):
    vals = [np.mean([float(r[f'{METRIC}_t{i}']) for r in agg[wv]])
            for i in range(N_SNAPSHOTS)]
    print(f'{wv:>8g} | ' + ' | '.join(f'{v:>9.4f}' for v in vals))

eik_dev by snapshot (mean over seeds)

   w_eik |         0 |       T/5 |      2T/5 |      3T/5 |      4T/5 |         T
-----------------------------------------------------------------------------------
       0 |    0.1218 |    0.1505 |    0.1720 |    0.1912 |    0.2032 |    0.2068
   1e-05 |    0.1408 |    0.1649 |    0.1858 |    0.2058 |    0.2247 |    0.2269
  0.0001 |    0.1191 |    0.1418 |    0.1611 |    0.1749 |    0.1855 |    0.1904
   0.001 |    0.1015 |    0.1117 |    0.1220 |    0.1290 |    0.1361 |    0.1426
    0.01 |    0.2843 |    0.0266 |    0.0201 |    0.0183 |    0.0163 |    0.0189
     0.1 |    0.2734 |    0.0084 |    0.0058 |    0.0062 |    0.0072 |    0.0104


In [9]:
# --- per-snapshot table ------------------------------------------------
METRIC = 'l2_abs'      # 'rel_L2' | 'vol_MAPE' | 'eik_dev' | 'l2_abs'

hdr = ' | '.join(f'{lab:>9}' for lab in tlabels)
print(f'{METRIC} by snapshot (mean over seeds)\n')
print(f"{'w_eik':>8} | {hdr}")
print('-' * (11 + 12 * len(tlabels)))
for wv in sorted(agg):
    vals = [np.mean([float(r[f'{METRIC}_t{i}']) for r in agg[wv]])
            for i in range(N_SNAPSHOTS)]
    print(f'{wv:>8g} | ' + ' | '.join(f'{v:>9.4f}' for v in vals))

l2_abs by snapshot (mean over seeds)

   w_eik |         0 |       T/5 |      2T/5 |      3T/5 |      4T/5 |         T
-----------------------------------------------------------------------------------
       0 |    0.0005 |    0.0081 |    0.0120 |    0.0126 |    0.0134 |    0.0144
   1e-05 |    0.0006 |    0.0078 |    0.0117 |    0.0121 |    0.0125 |    0.0141
  0.0001 |    0.0005 |    0.0059 |    0.0093 |    0.0092 |    0.0102 |    0.0123
   0.001 |    0.0006 |    0.0016 |    0.0018 |    0.0024 |    0.0025 |    0.0042
    0.01 |    0.0023 |    0.0046 |    0.0047 |    0.0047 |    0.0048 |    0.0051
     0.1 |    0.0022 |    0.0048 |    0.0049 |    0.0050 |    0.0049 |    0.0051
